# Subsample datasets to 100 per year

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import random
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Set up directories

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/complete_human/"

references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

# os.chdir(references)
# states_ref = pd.read_csv("states_ref.csv")

## Upload FASTAs

In [3]:
# Organize fastas

fastas = {}
for dirpath, dirs, files in os.walk(home):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = df_from_fasta(file_name)
            fastas[file_name.split("/")[-1]] = fasta
    break # Do not go into subfolders

In [4]:
print(fastas)

{'human_euro_H3_01-01-2000--12-31-2025.fasta':                                              full_header  \
0      >EPI_ISL_2460152|A/Belgium/IG0003/2020|H3N2|Be...   
1      >EPI_ISL_984703|A/Grenoble/2789/2020|H3N2|Gren...   
2      >EPI_ISL_526242|A/Netherlands/1797/2017|H3N2|N...   
3      >EPI_ISL_525546|A/Bari/532/2019|H3N2|Bari|2019...   
4      >EPI_ISL_525545|A/Bari/01/2020|H3N2|Bari|2020-...   
...                                                  ...   
76227  >EPI_ISL_163513|A/Franche_Comte/998/2014|H3N2|...   
76228  >EPI_ISL_163512|A/Haute_Normandie/1087/2014|H3...   
76229  >EPI_ISL_163509|A/Pays_de_Loire/1260/2014|H3N2...   
76230  >EPI_ISL_163508|A/Pays_de_Loire/1262/2014|H3N2...   
76231  >EPI_ISL_163506|A/Caen/349/2014|H3N2|Caen|2014...   

                                                sequence  
0      atgaagactatcattgctttgagctacattctatgtctggttttcg...  
1      atgaagactatcattgctttgagctacattctatgtctggttttcg...  
2      atgaagactatcattgctttgagctacattctatgtctggttttcg..

## Subsample

In [5]:
# Create a dictionary of dictionaries of dataframes grouped by year

fastas_grouped = {} # Overall dictionary -- length is number of datasets needed

for key in fastas:
    years = {} # For each dataset, there are a number of years
    fasta = fastas[key]
    fasta["Year"] = fasta["full_header"].apply(lambda x: dateutil.parser.parse(x.split("|")[-2]).year) # Find year
    for year, rows in fasta.groupby("Year"): # Separate dataframe into multiple dataframes by year
        years[year] = rows # For each year, there are a number of entries that have that year
    fastas_grouped[key] = years

print(fastas_grouped)

{'human_euro_H3_01-01-2000--12-31-2025.fasta': {2000:                                              full_header  \
65257  >EPI_ISL_1908|A/Madrid/SO2913/00|H3N2|Madrid|2...   
65258  >EPI_ISL_1893|A/Salamanca/RR682/00|H3N2|Salama...   
65259  >EPI_ISL_1892|A/Zaragoza/RR658/00|H3N2|Zaragoz...   
65260  >EPI_ISL_1891|A/Zaragoza/RR653/00|H3N2|Zaragoz...   
65261  >EPI_ISL_1890|A/Madrid/RR610/00|H3N2|Madrid|20...   
...                                                  ...   
73049  >EPI_ISL_187143|A/Bulgaria/874/2000|H3N2|Bulga...   
73053  >EPI_ISL_187057|A/Romania/427/2000|H3N2|Romani...   
73722  >EPI_ISL_188188|A/Romania/387/2000|H3N2|Romani...   
73727  >EPI_ISL_188150|A/Bulgaria/113/2000|H3N2|Bulga...   
75926  >EPI_ISL_127600|A/Ulan_Ude/01/2000|H3N2|Ulan_U...   

                                                sequence  Year  
65257  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  2000  
65258  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  2000  
65259  gcctcatccggcacactggagtt

In [6]:
subsampled_dfs = {} # Overall subsampled dictionary -- length is number of datasets needed
for key in fastas_grouped:
    dataset = fastas_grouped[key] # Dataset
    df = pd.DataFrame() # Hold subsampled data
    for year_key in dataset: # Dictionary of years and their dataframes 
        year_df = dataset[year_key] # One year and its data
        # If there are more than 100 entries, subsample a random 100 
        subsampled = year_df[["full_header", "sequence"]].sample(n=100, random_state=2009) if len(year_df) > 100 else year_df[["full_header", "sequence"]]
        # print(subsampled)
        df = pd.concat([df, subsampled]) # Add subsampled data to dataframe
    subsampled_dfs[key] = df # Add dataframe to dictionary of datasets

print(subsampled_dfs)
        

{'human_euro_H3_01-01-2000--12-31-2025.fasta':                                              full_header  \
65257  >EPI_ISL_1908|A/Madrid/SO2913/00|H3N2|Madrid|2...   
65258  >EPI_ISL_1893|A/Salamanca/RR682/00|H3N2|Salama...   
65259  >EPI_ISL_1892|A/Zaragoza/RR658/00|H3N2|Zaragoz...   
65260  >EPI_ISL_1891|A/Zaragoza/RR653/00|H3N2|Zaragoz...   
65261  >EPI_ISL_1890|A/Madrid/RR610/00|H3N2|Madrid|20...   
...                                                  ...   
57991  >EPI_ISL_20072221|A/Ireland/00036240/2025|H3N2...   
48380  >EPI_ISL_20283957|A/Oryol/RII-MH267394S/2025|H...   
53678  >EPI_ISL_20295641|A/England/01881771/2025|H3N2...   
53073  >EPI_ISL_20096672|A/Valencia/CSISP-11_0817/202...   
53688  >EPI_ISL_20295432|A/Netherlands/02472/2025|H3N...   

                                                sequence  
65257  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  
65258  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...  
65259  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga..

## De-duplicate

In [9]:
for key in subsampled_dfs:
    fasta_df = subsampled_dfs[key]
    fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
    fasta_df["Host"] = fasta_df["full_header"].apply(lambda x: "human" if "human" in x else x.split("/")[1] if "/" in x else "unknown") #  if len(x.split("/")) > 1 else "unknown")
    fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2] if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
    fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
    fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
    fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
    fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
    subsampled_dfs[key] = fasta_df.drop_duplicates(subset="Isolate", keep="first")
    print(fasta_df)

                                             full_header  \
65257  >EPI_ISL_1908|A/Madrid/SO2913/00|H3N2|Madrid|2...   
65258  >EPI_ISL_1893|A/Salamanca/RR682/00|H3N2|Salama...   
65259  >EPI_ISL_1892|A/Zaragoza/RR658/00|H3N2|Zaragoz...   
65260  >EPI_ISL_1891|A/Zaragoza/RR653/00|H3N2|Zaragoz...   
65261  >EPI_ISL_1890|A/Madrid/RR610/00|H3N2|Madrid|20...   
...                                                  ...   
57991  >EPI_ISL_20072221|A/Ireland/00036240/2025|H3N2...   
48380  >EPI_ISL_20283957|A/Oryol/RII-MH267394S/2025|H...   
53678  >EPI_ISL_20295641|A/England/01881771/2025|H3N2...   
53073  >EPI_ISL_20096672|A/Valencia/CSISP-11_0817/202...   
53688  >EPI_ISL_20295432|A/Netherlands/02472/2025|H3N...   

                                                sequence         Accession  \
65257  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...      EPI_ISL_1908   
65258  gcctcatccggcacactggagtttaacaatgaaagcttcaattgga...      EPI_ISL_1893   
65259  gcctcatccggcacactggagtttaacaatgaaagctt

## Download FASTAs

In [10]:
# Prepare for download
for key in subsampled_dfs:
    file_name = "subsampled_" + key # Create file name
    fasta = subsampled_dfs[key]
    df_to_fasta(fasta, file_name, home)
